# Phase 3 LangGraph HITL Workflow Demo

This notebook is a demo/education artifact for the Phase 3 LangGraph runtime workflow. It imports reusable workflow code from `agent_brain.orchestration.workflow` and does not duplicate orchestration logic in notebook cells.

No LLM calls are made. No OpenAI Agents SDK behavior is used. The notebook demonstrates deterministic LangGraph node execution, HITL pause/block behavior, and approved finalization.

## Prerequisites and run order

Before running this notebook, complete the Python setup in `docs/06-setup-runbook.md`:

1. Create a Python 3.11.x virtual environment in `agent-brain/`.
2. Install notebook dependencies with `python -m pip install -e .[dev,notebook]`.
3. Validate the LangGraph workflow with `python -m pytest tests/test_langgraph_workflow.py`.
4. Open this notebook with `jupyter lab notebooks/phase3-langgraph-hitl-demo.ipynb`.

This notebook does not require PostgreSQL, Neo4j, the mock pricing API, Phoenix, Langfuse, Microsoft Foundry Local, OpenAI, or OpenAI Agents SDK.

## Environment variables and local service assumptions

The examples use deterministic in-memory state and the LangGraph in-memory checkpointer. They do not call live services.

If you extend this notebook later to call the mock pricing API, keep that code behind documented setup steps and preserve the deterministic HITL checks shown here.

In [5]:
from IPython.display import HTML, display

display(HTML(
    """
    <style>
    div.output_area,
    div.output_subarea,
    div.output_text,
    div.jp-OutputArea,
    div.jp-OutputArea-child,
    div.jp-OutputArea-output,
    div.jp-RenderedText {
        max-width: 100% !important;
        overflow-x: hidden !important;
    }

    div.output_area pre,
    div.output_text pre,
    div.jp-OutputArea-output pre,
    div.jp-RenderedText pre {
        white-space: pre-wrap !important;
        overflow-wrap: anywhere !important;
        word-break: break-word !important;
        max-width: 100% !important;
        overflow-x: hidden !important;
    }
    </style>
    """
))

from pprint import pprint
from textwrap import fill

from agent_brain.governance.hitl import HITLDecisionOutcome
from agent_brain.orchestration.state import AgentBrainState, RetrievedContext, create_initial_state
from agent_brain.orchestration.workflow import (
    run_langgraph_workflow,
    workflow_state_from_agent_state,
)


def print_wrapped(value: object, width: int = 100) -> None:
    if value is None:
        print(value)
        return

    lines = str(value).splitlines() or [""]
    for line in lines:
        print(fill(line, width=width, break_long_words=True, break_on_hyphens=False))

## Build a deterministic high-risk workflow state

This state mirrors a governed renewal-review scenario. The risk tier and annual cost are deliberately high enough to require HITL before finalization.

In [6]:
high_risk_state = AgentBrainState(
    user_query="Should we renew OpenAI Enterprise?",
    retrieved_context=[
        RetrievedContext(
            vendor_name="OpenAI Enterprise",
            software_name="ChatGPT Enterprise",
            subscription_code="SUB-ENG-OPENAI-001",
            annual_cost_usd=43200.0,
            renewal_date="2026-10-15T00:00:00+00:00",
            risk_tier="HIGH",
            risk_category="DATA_RESIDENCY",
            risk_severity="HIGH",
            evidence_excerpt="cross-border processing evidence",
            source_document="openai-enterprise-sla.txt",
            priority_score=68.64,
        )
    ],
    trace_id="notebook-langgraph-hitl-demo",
)

workflow_state = workflow_state_from_agent_state(high_risk_state)
pprint(workflow_state, width=100, sort_dicts=False)

{'user_query': 'Should we renew OpenAI Enterprise?',
 'retrieved_context': [{'vendor_name': 'OpenAI Enterprise',
                        'software_name': 'ChatGPT Enterprise',
                        'subscription_code': 'SUB-ENG-OPENAI-001',
                        'annual_cost_usd': 43200.0,
                        'renewal_date': '2026-10-15T00:00:00+00:00',
                        'risk_tier': 'HIGH',
                        'risk_category': 'DATA_RESIDENCY',
                        'risk_severity': 'HIGH',
                        'evidence_excerpt': 'cross-border processing evidence',
                        'source_document': 'openai-enterprise-sla.txt',
                        'priority_score': 68.64}],
 'compliance_risks': [],
 'live_pricing': [],
 'recommendation_draft': None,
 'human_approval_status': 'NOT_REQUIRED',
 'final_output': None,
 'trace_id': 'notebook-langgraph-hitl-demo',
 'safety_flags': []}


## Run the LangGraph workflow before approval

The workflow drafts a deterministic recommendation and routes to HITL. The final output remains empty because no approval decision has been supplied.

In [7]:
hitl_required_result = run_langgraph_workflow(
    workflow_state,
    thread_id="notebook-hitl-required",
)

print_wrapped(hitl_required_result["workflow_status"])
print_wrapped(hitl_required_result["final_output"])
pprint(hitl_required_result["hitl_pause"], width=100, sort_dicts=False)
assert hitl_required_result["workflow_status"] == "HITL_REQUIRED"
assert hitl_required_result["final_output"] is None

HITL_REQUIRED
None
{'required': True,
 'reason': 'Human approval required before finalization.',
 'draft_summary': 'Analyzed 1 retrieved context rows, 0 compliance risks, and 0 live pricing '
                  'records.',
 'recommended_action': 'Prepare renewal review for human approval',
 'trace_id': 'notebook-langgraph-hitl-demo',
 'safety_flags': ['HITL_REQUIRED']}


## Resume with an approved human decision

The notebook now supplies a serializable human approval decision. This is still deterministic application logic; no model is deciding whether finalization is allowed.

In [8]:
approved_state = dict(workflow_state)
approved_state["finalization_decision"] = {
    "outcome": HITLDecisionOutcome.APPROVED.value,
    "reviewer": "Compliance reviewer",
    "rationale": "Approved for governed renewal review.",
}

approved_result = run_langgraph_workflow(
    approved_state,
    thread_id="notebook-hitl-approved",
)

print_wrapped(approved_result["workflow_status"])
print_wrapped(approved_result["final_output"])
assert approved_result["workflow_status"] == "FINALIZED_WITH_HITL"
assert approved_result["final_output"] is not None

FINALIZED_WITH_HITL
Final recommendation: Prepare renewal review for human approval. Highest priority score is 68.64;
maximum annual exposure is $43,200.00; high-risk contexts=1; high-severity risks=0; live pricing
available=False. Human approval is required before finalizing renewal or cancellation action.
Approved by Compliance reviewer.


## Run a low-risk workflow that finalizes without HITL

A neutral governance-summary request has no cancellation or renewal action and no high-risk context, so the workflow can finalize without HITL.

In [9]:
low_risk_state = workflow_state_from_agent_state(
    create_initial_state("Summarize governance posture.")
)

low_risk_result = run_langgraph_workflow(
    low_risk_state,
    thread_id="notebook-no-hitl-required",
)

print_wrapped(low_risk_result["workflow_status"])
print_wrapped(low_risk_result["final_output"])
assert low_risk_result["workflow_status"] == "FINALIZED_WITHOUT_HITL"
assert low_risk_result["final_output"] is not None

FINALIZED_WITHOUT_HITL
Final recommendation: Gather pricing before recommendation. Highest priority score is 0.00; maximum
annual exposure is $0.00; high-risk contexts=0; high-severity risks=0; live pricing available=False.


## Output interpretation

The key proof points are:

- `HITL_REQUIRED` appears before approval for high-risk renewal review.
- `final_output` is `None` before approval.
- `FINALIZED_WITH_HITL` appears only after an approved human decision.
- `FINALIZED_WITHOUT_HITL` appears for low-risk non-renewal/non-cancellation summary work.

The notebook demonstrates workflow behavior only. The authoritative implementation remains in `agent_brain.orchestration.workflow`, `agent_brain.orchestration.recommendation`, and `agent_brain.governance.hitl`.

## Limitations and reset instructions

This notebook uses in-memory workflow state and in-memory LangGraph checkpointing. Restarting the kernel resets notebook state.

No database rows, Neo4j graph nodes, pricing API state, Phoenix traces, Langfuse events, or audit records are written by this notebook. If future versions add live service calls, update `docs/06-setup-runbook.md` before treating the notebook as end-user ready.